# Colab Ollama Qwen 27B Q4_K_M host

Runs chess-coach web app on Colab GPU with code from GitHub and model served by Ollama. No Hugging Face token required for the default public Ollama model.

## Required settings

Set before running cells if needed:

- `CHESS_REPO_URL`: GitHub repo URL.
- `CHESS_BRANCH`: branch to test, default `feat/chess-coach-sft`.
- `OLLAMA_MODEL`: default `qwen3.6:27b-q4_K_M`.
- `CHESS_HOST`: default `127.0.0.1`.
- `CHESS_PORT`: default `7860`.

In [ ]:
import json, os, pathlib, subprocess, time, urllib.request

DEFAULT_ENV = {
    'CHESS_REPO_URL': 'https://github.com/RyanDev1st/llm-and-engine.git',
    'CHESS_BRANCH': 'feat/chess-coach-sft',
    'OLLAMA_MODEL': 'qwen3.6:27b-q4_K_M',
    'CHESS_HOST': '127.0.0.1',
    'CHESS_PORT': '7860',
}

for key, value in DEFAULT_ENV.items():
    os.environ.setdefault(key, value)

def run(cmd, **kwargs):
    print('>', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=True, text=True, **kwargs)

def out(cmd):
    return subprocess.check_output(cmd, text=True).strip()

def env_required(name):
    value = os.environ.get(name, '').strip()
    if not value:
        raise RuntimeError(f'Set {name} before running this notebook')
    return value

In [ ]:
gpu_csv = out(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'])
name, total_mb, free_mb = [x.strip() for x in gpu_csv.split(',', 2)]
total_mb, free_mb = int(total_mb), int(free_mb)
print({'gpu': name, 'total_mb': total_mb, 'free_mb': free_mb})
if total_mb < 30000:
    raise RuntimeError('27B Q4 target expects high-VRAM GPU. Prefer H100/A100 before continuing.')


In [ ]:
REPO_URL = env_required('CHESS_REPO_URL')
BRANCH = os.environ.get('CHESS_BRANCH', 'feat/chess-coach-sft')
WORKDIR = pathlib.Path('/content/llm_tool_calling_research_workspace')
if WORKDIR.exists():
    run(['git', '-C', str(WORKDIR), 'fetch', 'origin', BRANCH])
    run(['git', '-C', str(WORKDIR), 'checkout', BRANCH])
    run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'])
else:
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(WORKDIR)])
commit = out(['git', '-C', str(WORKDIR), 'rev-parse', 'HEAD'])
status = out(['git', '-C', str(WORKDIR), 'status', '--short'])
print({'branch': BRANCH, 'commit': commit, 'dirty': bool(status)})


In [ ]:
run(['curl', '-fsSL', 'https://ollama.com/install.sh'], stdout=open('/tmp/ollama-install.sh', 'w'))
run(['sh', '/tmp/ollama-install.sh'])


In [ ]:
ollama_log = open('/tmp/ollama.log', 'w')
ollama = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT, text=True)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2).read()
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('ollama did not start')
OLLAMA_MODEL = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b-q4_K_M')
run(['ollama', 'pull', OLLAMA_MODEL])
print({'ollama_model': OLLAMA_MODEL})


In [ ]:
env = os.environ.copy()
env.update({
    'PYTHONPATH': str(WORKDIR / 'src' / 'llm'),
    'CHESS_HOST': os.environ.get('CHESS_HOST', '127.0.0.1'),
    'CHESS_PORT': os.environ.get('CHESS_PORT', '7860'),
    'OLLAMA_MODEL': OLLAMA_MODEL,
    'OLLAMA_HOST': 'http://127.0.0.1:11434',
    'CHESS_GGUF_PATH': '/content/models/not-used.gguf',
})
print({k: env[k] for k in ['CHESS_HOST', 'CHESS_PORT', 'OLLAMA_MODEL', 'OLLAMA_HOST']})
server = subprocess.Popen(['python', '-m', 'backend.server'], cwd=str(WORKDIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
ready = False
for _ in range(180):
    line = server.stdout.readline().strip()
    if line:
        print(line)
    if server.poll() is not None:
        raise RuntimeError(f'server exited early with code {server.returncode}')
    if 'open http://' in line:
        ready = True
        break
if not ready:
    raise RuntimeError('server did not report ready')


In [ ]:
state_body = None
for _ in range(60):
    try:
        state_body = urllib.request.urlopen('http://127.0.0.1:7860/api/state', timeout=2).read().decode()
        break
    except Exception:
        time.sleep(2)
if state_body is None:
    raise RuntimeError('server did not answer /api/state')
print(state_body[:1000])


In [ ]:
run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared'])
run(['chmod', '+x', '/usr/local/bin/cloudflared'])
print('Tunnel URL below is public. Stop cell when done testing.')
subprocess.run(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'], check=True)
